### Set up

In [3]:
pip install pandas ; plotly ; kaleido

Note: you may need to restart the kernel to use updated packages.


ERROR: Invalid requirement: ''

[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: C:\Users\dorvi\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [5]:
%pip install pandas
%pip install plotly
%pip install kaleido
# 3_data_exploration/day5_exploration.py
# Day 5 – Data Exploration (VS Code version)
# Produces readable labels + polished visuals (PNG + HTML) under 4_outputs/

import os
import re
import pandas as pd
import plotly.express as px
import plotly.io as pio

# ----------------------------
# Paths (adjust if your repo differs)
# ----------------------------
CLEAN_PATH = "1_datasets/cleaned/hospital_clean.csv"
LONG_PATH  = "1_datasets/cleaned/hospital_measures_long.csv"
OUT_DIR    = "4_outputs"
os.makedirs(OUT_DIR, exist_ok=True)


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: C:\Users\dorvi\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: C:\Users\dorvi\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: C:\Users\dorvi\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
# ----------------------------
# 1) Load data
# ----------------------------
import pandas as pd

# Define paths
CLEAN_PATH = "../1_datasets/cleaned/hospital_clean.csv"
LONG_PATH  = "../1_datasets/cleaned/hospital_measures_long.csv"

# Load using the path variables (NOT the filenames directly!)
clean = pd.read_csv(CLEAN_PATH)
long  = pd.read_csv(LONG_PATH)

# Display first 2 rows
print("Hospital Clean Data:")
display(clean.head(2))
print("\nHospital Measures Long Data:")
display(long.head(2))

# ----------------------------
# Safety guard: Filter to only your 4 target hospitals
# ----------------------------
target_names = {
    "MANATEE MEMORIAL HOSPITAL",
    "SARASOTA MEMORIAL HOSPITAL",
    "HCA FLORIDA BLAKE HOSPITAL",
    "LAKEWOOD RANCH MEDICAL CENTER",
}

clean = clean[clean["Hospital Name"].isin(target_names)].copy()
long  = long[long["Hospital Name"].isin(target_names)].copy()

print(f"\n✅ Filtered to {len(clean)} hospitals in clean data")
print(f"✅ Filtered to {len(long)} records in measures data")

Hospital Clean Data:


,Provider ID,Hospital Name,City,State,ZIP Code,Readmission_Score_mean,Readmission_Score_median,Readmission_n,HCAHPS_LinearMean_mean,HCAHPS_Star_mean,HCAHPS_ResponseRate_mean,HCAHPS_n,Safety_Score_mean,Safety_n
0,100035,MANATEE MEMORIAL HOSPITAL,BRADENTON,FL,34208,0.9975,0.98685,6,79.8,1.818182,18.0,93,12.8200,20
1,100087,SARASOTA MEMORIAL HOSPITAL,SARASOTA,FL,34239,0.9202,0.94225,6,84.2,2.909091,28.0,93,9.6685,20



Hospital Measures Long Data:


,Provider ID,Measure,Score,Domain,Hospital Name,City,State,ZIP Code
0,100035,READM-30-PN-HRRP,0.9923,Readmissions,MANATEE MEMORIAL HOSPITAL,BRADENTON,FL,34208
1,100035,READM-30-HIP-KNEE-HRRP,0.9761,Readmissions,MANATEE MEMORIAL HOSPITAL,BRADENTON,FL,34208



✅ Filtered to 4 hospitals in clean data
✅ Filtered to 476 records in measures data


In [3]:
# ----------------------------
# 1.1) Quick Data Overview
# ----------------------------

# See what columns you have
print("\n📊 Available columns in clean data:")
print(clean.columns.tolist())

print("\n📊 Available columns in measures data:")
print(long.columns.tolist())

# Check what types of measures you have (FIXED - use 'Measure' not 'Measure Name')
print("\n📋 Types of measures available:")
print(long['Measure'].value_counts())

# See the hospitals
print("\n🏥 Hospitals in dataset:")
print(clean['Hospital Name'].tolist())


📊 Available columns in clean data:
['Provider ID', 'Hospital Name', 'City', 'State', 'ZIP Code', 'Readmission_Score_mean', 'Readmission_Score_median', 'Readmission_n', 'HCAHPS_LinearMean_mean', 'HCAHPS_Star_mean', 'HCAHPS_ResponseRate_mean', 'HCAHPS_n', 'Safety_Score_mean', 'Safety_n']

📊 Available columns in measures data:
['Provider ID', 'Measure', 'Score', 'Domain', 'Hospital Name', 'City', 'State', 'ZIP Code']

📋 Types of measures available:
Measure
READM-30-PN-HRRP                                                    4
READM-30-HIP-KNEE-HRRP                                              4
READM-30-HF-HRRP                                                    4
READM-30-COPD-HRRP                                                  4
READM-30-CABG-HRRP                                                  4
                                                                   ..
Perioperative pulmonary embolism or deep vein thrombosis rate       4
Postoperative sepsis rate                          

In [4]:
# ----------------------------
# 2) Build a human-readable labeler for HCAHPS measures
#    Works for LINEAR_SCORE / STAR_RATING / *_A_P / *_U_P / *_SN_P / *_P
# ----------------------------
import re  # ← Add this import!

PREFIX_MAP = {
    "H_COMP_1": "Nurse communication",
    "H_COMP_2": "Doctor communication",
    "H_COMP_3": "Responsiveness of hospital staff",
    "H_COMP_5": "Communication about medicines",
    "H_COMP_6": "Discharge information",
    "H_COMP_7": "Care transition",
    "H_CLEAN_HSP": "Hospital kept clean",
    "H_QUIET_HSP": "Hospital kept quiet",
    "H_HSP_RATING": "Overall hospital rating",
    "H_RECMND": "Would recommend hospital",
    # Common detailed items seen in public HCAHPS files:
    "H_NURSE_RESPECT": "Nurses treated patient with courtesy & respect",
    "H_NURSE_LISTEN": "Nurses listened carefully",
    "H_NURSE_EXPLAIN": "Nurses explained things clearly",
    "H_DOCTOR_RESPECT": "Doctors treated patient with courtesy & respect",
    "H_DOCTOR_LISTEN": "Doctors listened carefully",
    "H_DOCTOR_EXPLAIN": "Doctors explained things clearly",
    "H_CALL_BUTTON": "Help as soon as wanted after call button",
    "H_BATH_HELP": "Help to bathroom or bedpan as soon as wanted",
    "H_MED_EXPLAIN": "Staff clearly explained what a new medicine was for",
    "H_MED_SIDE_EFFECTS": "Staff described possible side effects of new medicines",
    "H_DISCHARGE_INFO": "Patient got information about recovery at home",
}

SUFFIX_MAP = {
    "LINEAR_SCORE": "(Linear score)",
    "STAR_RATING": "(Star rating)",
    "A_P": '— "Always" (%)',
    "U_P": '— "Usually" (%)',
    "SN_P": '— "Sometimes/Never" (%)',
    "P": "(percent)",
}

def pretty_from_code(code: str) -> str:
    """
    Convert HCAHPS measure codes into human-readable labels.
    Example: H_COMP_1_LINEAR_SCORE → "Nurse communication (Linear score)"
    """
    # 1) Split into prefix + suffix by matching a known suffix at the end
    suffix_key = None
    for suf in ["LINEAR_SCORE", "STAR_RATING", "A_P", "U_P", "SN_P", "P"]:
        if code.endswith("_" + suf) or code.endswith(suf):
            suffix_key = suf
            break

    base = code
    if suffix_key:
        base = re.sub(rf"(_{suffix_key})$", "", code)  # drop trailing suffix

    # 2) Map the prefix by longest match (e.g., H_NURSE_RESPECT before H_NURSE)
    prefix_label = None
    for k in sorted(PREFIX_MAP.keys(), key=len, reverse=True):
        if base.startswith(k):
            prefix_label = PREFIX_MAP[k]
            break

    # If still unknown, try generic transformation: H_ABC_DEF -> "Abc def"
    if not prefix_label:
        generic = base
        if generic.startswith("H_"):
            generic = generic[2:]
        generic = generic.replace("_", " ").title()
        prefix_label = generic

    # 3) Attach suffix meaning
    if suffix_key:
        suffix_text = SUFFIX_MAP.get(suffix_key, f"({suffix_key.replace('_',' ').title()})")
        return f"{prefix_label} {suffix_text}"
    else:
        return prefix_label

# Apply to long dataframe
if "Measure" in long.columns:
    long["Measure_pretty"] = long["Measure"].apply(pretty_from_code)
    print("✅ Created 'Measure_pretty' column")
    
    # Show a sample of the transformation
    print("\n🔍 Sample of pretty labels:")
    sample = long[['Measure', 'Measure_pretty']].drop_duplicates().head(10)
    display(sample)
else:
    raise ValueError("Expected column 'Measure' in hospital_measures_long.csv")

✅ Created 'Measure_pretty' column

🔍 Sample of pretty labels:


,Measure,Measure_pretty
0,READM-30-PN-HRRP,Readm-30-Pn-Hrrp (percent)
1,READM-30-HIP-KNEE-HRRP,Readm-30-Hip-Knee-Hrrp (percent)
2,READM-30-HF-HRRP,Readm-30-Hf-Hrrp (percent)
3,READM-30-COPD-HRRP,Readm-30-Copd-Hrrp (percent)
4,READM-30-CABG-HRRP,Readm-30-Cabg-Hrrp (percent)
5,READM-30-AMI-HRRP,Readm-30-Ami-Hrrp (percent)
24,H_COMP_1_A_P,"Nurse communication — ""Always"" (%)"
25,H_COMP_1_SN_P,"Nurse communication — ""Sometimes/Never"" (%)"
26,H_COMP_1_U_P,"Nurse communication — ""Usually"" (%)"
27,H_COMP_1_LINEAR_SCORE,Nurse communication (Linear score)


In [5]:
"""
Complete PREFIX_MAP for Hospital Performance Measures
Makes charts beautiful and presentation-ready! 🎨
"""

PREFIX_MAP = {
    # ========================================
    # READMISSION MEASURES (6 measures)
    # ========================================
    "READM-30-AMI-HRRP": "Heart Attack - 30-Day Readmission",
    "READM-30-CABG-HRRP": "Heart Bypass Surgery - 30-Day Readmission",
    "READM-30-COPD-HRRP": "COPD - 30-Day Readmission",
    "READM-30-HF-HRRP": "Heart Failure - 30-Day Readmission",
    "READM-30-HIP-KNEE-HRRP": "Hip/Knee Replacement - 30-Day Readmission",
    "READM-30-PN-HRRP": "Pneumonia - 30-Day Readmission",
    
    # ========================================
    # MORTALITY/DEATH RATE MEASURES (9 measures)
    # ========================================
    "Death rate for heart attack patients": "Mortality Rate - Heart Attack",
    "Death rate for heart failure patients": "Mortality Rate - Heart Failure",
    "Death rate for pneumonia patients": "Mortality Rate - Pneumonia",
    "Death rate for COPD patients": "Mortality Rate - COPD",
    "Death rate for stroke patients": "Mortality Rate - Stroke",
    "Death rate for CABG surgery patients": "Mortality Rate - Heart Bypass Surgery",
    "Death rate among surgical inpatients with serious treatable complications": "Mortality Rate - Surgical Complications",
    "Hybrid Hospital-Wide All-Cause Risk Standardized Mortality Rate": "Overall Hospital Mortality Rate",
    
    # ========================================
    # PATIENT SAFETY & COMPLICATIONS (15 measures)
    # ========================================
    "CMS Medicare PSI 90: Patient safety and adverse events composite": "Overall Patient Safety Score",
    "Rate of complications for hip/knee replacement patients": "Hip/Knee Surgery - Complication Rate",
    "Abdominopelvic accidental puncture or laceration rate": "Surgical Injury - Accidental Cut/Puncture",
    "Iatrogenic pneumothorax rate": "Collapsed Lung During Treatment",
    "In-hospital fall-associated fracture rate": "Patient Fall - Resulting in Fracture",
    "Pressure ulcer rate": "Bedsore Development Rate",
    "Perioperative pulmonary embolism or deep vein thrombosis rate": "Surgery - Blood Clot Formation",
    "Postoperative acute kidney injury requiring dialysis rate": "Post-Surgery - Kidney Failure",
    "Postoperative hemorrhage or hematoma rate": "Post-Surgery - Bleeding/Bruising",
    "Postoperative respiratory failure rate": "Post-Surgery - Breathing Failure",
    "Postoperative sepsis rate": "Post-Surgery - Blood Infection",
    "Postoperative wound dehiscence rate": "Post-Surgery - Wound Reopening",
    
    # ========================================
    # HCAHPS: NURSE COMMUNICATION (9 measures)
    # ========================================
    "H_COMP_1": "Nurse Communication",
    "H_NURSE_RESPECT": "Nurses - Courtesy & Respect",
    "H_NURSE_LISTEN": "Nurses - Listened Carefully",
    "H_NURSE_EXPLAIN": "Nurses - Explained Clearly",
    
    # ========================================
    # HCAHPS: DOCTOR COMMUNICATION (9 measures)
    # ========================================
    "H_COMP_2": "Doctor Communication",
    "H_DOCTOR_RESPECT": "Doctors - Courtesy & Respect",
    "H_DOCTOR_LISTEN": "Doctors - Listened Carefully",
    "H_DOCTOR_EXPLAIN": "Doctors - Explained Clearly",
    
    # ========================================
    # HCAHPS: RESPONSIVENESS (9 measures)
    # ========================================
    "H_COMP_3": "Staff Responsiveness",
    "H_CALL_BUTTON": "Response to Call Button",
    "H_BATH_HELP": "Help with Bathroom",
    
    # ========================================
    # HCAHPS: MEDICATION COMMUNICATION (9 measures)
    # ========================================
    "H_COMP_5": "Medication Communication",
    "H_MED_FOR": "Staff Explained New Medications",
    "H_SIDE_EFFECTS": "Staff Explained Side Effects",
    
    # ========================================
    # HCAHPS: DISCHARGE INFORMATION (6 measures)
    # ========================================
    "H_COMP_6": "Discharge Information",
    "H_DISCH_HELP": "Received Help After Discharge",
    "H_SYMPTOMS": "Understood Symptoms to Watch",
    
    # ========================================
    # HCAHPS: CARE TRANSITION (12 measures)
    # ========================================
    "H_COMP_7": "Care Transition",
    "H_CT_UNDER": "Care Transition - Understood Care Plan",
    "H_CT_MED": "Care Transition - Medication Management",
    "H_CT_PREFER": "Care Transition - Preferences Considered",
    
    # ========================================
    # HCAHPS: CLEANLINESS (5 measures)
    # ========================================
    "H_CLEAN_HSP": "Hospital Cleanliness",
    "H_CLEAN": "Hospital Cleanliness",
    
    # ========================================
    # HCAHPS: QUIETNESS (5 measures)
    # ========================================
    "H_QUIET_HSP": "Hospital Quietness at Night",
    "H_QUIET": "Hospital Quietness at Night",
    
    # ========================================
    # HCAHPS: OVERALL RATING (5 measures)
    # ========================================
    "H_HSP_RATING": "Overall Hospital Rating",
    
    # ========================================
    # HCAHPS: RECOMMENDATION (5 measures)
    # ========================================
    "H_RECMND": "Would Recommend Hospital",
    
    # ========================================
    # HCAHPS: OVERALL STAR (1 measure)
    # ========================================
    "H_STAR_RATING": "Overall Hospital Star Rating",
}

SUFFIX_MAP = {
    "LINEAR_SCORE": "(Score)",
    "STAR_RATING": "(Stars)",
    "A_P": '- Always',
    "U_P": '- Usually',
    "SN_P": '- Sometimes/Never',
    "N_P": '- No',
    "Y_P": '- Yes',
    "P": "(%)",
    "HRRP": "Rate",
    # Care Transition specific suffixes
    "A": "- Agree",
    "SA": "- Strongly Agree",
    "D_SD": "- Disagree/Strongly Disagree",
    # Recommendation specific suffixes
    "DY": "- Definitely Yes",
    "PY": "- Probably Yes",
    "DN": "- Definitely/Probably No",
    # Rating specific suffixes
    "9_10": "- Rating 9-10 (Best)",
    "7_8": "- Rating 7-8",
    "0_6": "- Rating 0-6 (Worst)",
}

def pretty_from_code(code: str) -> str:
    """
    Convert measure codes into beautiful, presentation-ready labels.
    
    Examples:
    - H_COMP_1_A_P → "Nurse Communication - Always"
    - READM-30-PN-HRRP → "Pneumonia - 30-Day Readmission Rate"
    - Death rate for pneumonia patients → "Mortality Rate - Pneumonia"
    """
    import re
    
    # Handle measures that are already human-readable (like death rates)
    if code in PREFIX_MAP:
        return PREFIX_MAP[code]
    
    # 1) Find and extract suffix
    suffix_key = None
    for suf in sorted(SUFFIX_MAP.keys(), key=len, reverse=True):
        if code.endswith("_" + suf) or code.endswith(suf):
            suffix_key = suf
            break
    
    # Remove suffix to get base code
    base = code
    if suffix_key:
        base = re.sub(rf"(_{suffix_key})$", "", code)
    
    # 2) Find matching prefix (longest match first)
    prefix_label = None
    for k in sorted(PREFIX_MAP.keys(), key=len, reverse=True):
        if base.startswith(k) or base == k:
            prefix_label = PREFIX_MAP[k]
            break
    
    # Fallback: generic transformation if not found
    if not prefix_label:
        generic = base
        if generic.startswith("H_"):
            generic = generic[2:]
        generic = generic.replace("_", " ").replace("-", " ").title()
        prefix_label = generic
    
    # 3) Attach suffix meaning
    if suffix_key:
        suffix_text = SUFFIX_MAP.get(suffix_key, f" ({suffix_key})")
        return f"{prefix_label} {suffix_text}"
    else:
        return prefix_label


# ========================================
# USAGE EXAMPLE
# ========================================
if __name__ == "__main__":
    # Test the function
    test_codes = [
        "READM-30-PN-HRRP",
        "H_COMP_1_A_P",
        "H_NURSE_EXPLAIN_LINEAR_SCORE",
        "Death rate for pneumonia patients",
        "CMS Medicare PSI 90: Patient safety and adverse events composite",
        "H_HSP_RATING_9_10",
    ]
    
    print("=" * 70)
    print("TESTING MEASURE LABEL TRANSFORMATIONS")
    print("=" * 70)
    for code in test_codes:
        print(f"{code:50s} → {pretty_from_code(code)}")

TESTING MEASURE LABEL TRANSFORMATIONS
READM-30-PN-HRRP                                   → Pneumonia - 30-Day Readmission
H_COMP_1_A_P                                       → Nurse Communication - Always
H_NURSE_EXPLAIN_LINEAR_SCORE                       → Nurses - Explained Clearly (Score)
Death rate for pneumonia patients                  → Mortality Rate - Pneumonia
CMS Medicare PSI 90: Patient safety and adverse events composite → Overall Patient Safety Score
H_HSP_RATING_9_10                                  → Overall Hospital Rating - Rating 9-10 (Best)


In [6]:
# Apply the new mapping
long["Measure_pretty"] = long["Measure"].apply(pretty_from_code)

print("✅ Created beautiful measure labels!")
print("\n🔍 Sample transformations:")
display(long[['Measure', 'Measure_pretty']].drop_duplicates().head(15))

✅ Created beautiful measure labels!

🔍 Sample transformations:


,Measure,Measure_pretty
0,READM-30-PN-HRRP,Pneumonia - 30-Day Readmission
1,READM-30-HIP-KNEE-HRRP,Hip/Knee Replacement - 30-Day Readmission
2,READM-30-HF-HRRP,Heart Failure - 30-Day Readmission
3,READM-30-COPD-HRRP,COPD - 30-Day Readmission
4,READM-30-CABG-HRRP,Heart Bypass Surgery - 30-Day Readmission
5,READM-30-AMI-HRRP,Heart Attack - 30-Day Readmission
24,H_COMP_1_A_P,Nurse Communication - Always
25,H_COMP_1_SN_P,Nurse Communication - Sometimes/Never
26,H_COMP_1_U_P,Nurse Communication - Usually
27,H_COMP_1_LINEAR_SCORE,Nurse Communication (Score)


In [ ]:
## Visuals

In [37]:
%pip install nbformat plotly kaleido

   ---------------------------------------- 0.0/78.5 kB ? eta -:--:--
   --------------- ------------------------ 30.7/78.5 kB 640.0 kB/s eta 0:00:01
   ---------------------------------------- 78.5/78.5 kB 863.5 kB/s eta 0:00:00
   ---------------------------------------- 0.0/90.0 kB ? eta -:--:--
   ---------------------------------------- 90.0/90.0 kB 2.6 MB/s eta 0:00:00
   ---------------------------------------- 0.0/67.6 kB ? eta -:--:--
   ---------------------------------------- 67.6/67.6 kB 1.9 MB/s eta 0:00:00
   ---------------------------------------- 0.0/224.0 kB ? eta -:--:--
   ---------------------------------------- 224.0/224.0 kB 4.5 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: C:\Users\dorvi\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [13]:
pio.renderers.default = "plotly_mimetype+notebook_connected"

In [24]:
"""
Hospital Performance Visualization Suite
5 Beautiful, Presentation-Ready Charts 📊✨
"""

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

def save_chart(fig, base_name, width=1200, height=700, scale=2):
    html_path = OUT_DIR / f"{base_name}.html"
    png_path  = OUT_DIR / f"{base_name}.png"

    fig.write_html(str(html_path), include_plotlyjs="cdn")

    try:
        fig.write_image(str(png_path), format="png", width=width, height=height, scale=scale)
    except Exception as e:
        print(f"⚠️ PNG export failed for {base_name}. Error: {e}")

    print(f"💾 Saved: {html_path.name}, {png_path.name}")

# ============================================================

# ============================================================
# Configure Plotly to display
pio.renderers.default = "plotly_mimetype+notebook_connected"
print("✅ Plotly configured!\n")

# ============================================================
# CONSISTENT COLOR SCHEME FOR ALL HOSPITALS
# ============================================================
HOSPITAL_COLORS = {
    "MANATEE MEMORIAL HOSPITAL": "#FF6B6B",      # Coral Red
    "SARASOTA MEMORIAL HOSPITAL": "#4ECDC4",     # Turquoise
    "HCA FLORIDA BLAKE HOSPITAL": "#45B7D1",     # Sky Blue
    "LAKEWOOD RANCH MEDICAL CENTER": "#FFA07A",  # Light Salmon
}

# Shorter names for better chart readability
HOSPITAL_SHORT_NAMES = {
    "MANATEE MEMORIAL HOSPITAL": "Manatee Memorial",
    "SARASOTA MEMORIAL HOSPITAL": "Sarasota Memorial",
    "HCA FLORIDA BLAKE HOSPITAL": "Blake Hospital",
    "LAKEWOOD RANCH MEDICAL CENTER": "Lakewood Ranch",
}

# Apply short names to dataframes
clean['Hospital_Short'] = clean['Hospital Name'].map(HOSPITAL_SHORT_NAMES)
long['Hospital_Short'] = long['Hospital Name'].map(HOSPITAL_SHORT_NAMES)

print("🎨 Hospital Color Coding:")
for hospital, color in HOSPITAL_COLORS.items():
    short_name = HOSPITAL_SHORT_NAMES[hospital]
    print(f"   {color} ● {short_name}")
print()


# ============================================================
# 1️⃣ OVERALL COMPARISON CHART
# Shows all 4 hospitals comparing Readmissions, HCAHPS, and Safety
# ============================================================

print("=" * 70)
print("📊 Creating Chart 1: Overall Hospital Comparison")
print("=" * 70)

# Use the aggregated data from 'clean' dataframe
comparison_data = clean[['Hospital_Short', 'Readmission_Score_mean', 
                         'HCAHPS_LinearMean_mean', 'Safety_Score_mean']].copy()

# Rename columns for better labels
comparison_data.columns = ['Hospital', 'Readmissions', 'Patient Experience', 'Safety']

# Melt for grouped bar chart
comparison_melted = comparison_data.melt(id_vars='Hospital', 
                                         var_name='Category', 
                                         value_name='Score')

# Create grouped bar chart with hospital colors
fig1 = px.bar(comparison_melted,
              x='Category',
              y='Score',
              color='Hospital',
              barmode='group',
              title='🏥 Overall Hospital Performance Comparison',
              labels={'Score': 'Average Score', 'Category': 'Performance Category'},
              color_discrete_map={v: HOSPITAL_COLORS[k] for k, v in HOSPITAL_SHORT_NAMES.items()},
              height=500)

fig1.update_layout(
    font=dict(size=12),
    title_font_size=20,
    title_x=0.5,
    legend=dict(title="Hospital", orientation="v", yanchor="top", y=1, xanchor="left", x=1.02),
    hovermode='x unified'
)

fig1.update_traces(texttemplate='%{y:.1f}', textposition='outside')

fig1.show()
save_chart(fig1, "01_overall_comparison")
print("✅ Chart 1 complete!\n")


# ============================================================
# 2️⃣ TOP 5 HCAHPS QUESTIONS (Overall)
# Highest scoring patient satisfaction measures
# ============================================================

print("=" * 70)
print("📈 Creating Chart 2: Top 5 HCAHPS Questions")
print("=" * 70)

# Filter HCAHPS measures (those starting with H_)
hcahps_data = long[long['Measure'].str.startswith('H_', na=False)].copy()

# Calculate average score per measure across all hospitals
hcahps_avg = hcahps_data.groupby(['Measure_pretty', 'Hospital_Short'])['Score'].mean().reset_index()
top5_measures = hcahps_data.groupby('Measure_pretty')['Score'].mean().nlargest(5).index

top5_data = hcahps_avg[hcahps_avg['Measure_pretty'].isin(top5_measures)]

# Create horizontal bar chart with hospital colors
fig2 = px.bar(top5_data,
              y='Measure_pretty',
              x='Score',
              color='Hospital_Short',
              orientation='h',
              title='📈 Top 5 Patient Satisfaction Measures',
              labels={'Score': 'Average Score', 'Measure_pretty': ''},
              color_discrete_map={v: HOSPITAL_COLORS[k] for k, v in HOSPITAL_SHORT_NAMES.items()},
              barmode='group',
              height=500)

fig2.update_layout(
    title_font_size=20,
    title_x=0.5,
    font=dict(size=11),
    legend=dict(title="Hospital"),
    xaxis_title='Average Score'
)

fig2.update_traces(texttemplate='%{x:.1f}', textposition='outside')

fig2.show()
save_chart(fig2, "02_top5_hcahps")
print("✅ Chart 2 complete!\n")


# ============================================================
# 3️⃣ BOTTOM 5 HCAHPS QUESTIONS (Overall)
# Lowest scoring patient satisfaction measures
# ============================================================

print("=" * 70)
print("📉 Creating Chart 3: Bottom 5 HCAHPS Questions")
print("=" * 70)

# Get bottom 5 HCAHPS measures
bottom5_measures = hcahps_data.groupby('Measure_pretty')['Score'].mean().nsmallest(5).index
bottom5_data = hcahps_avg[hcahps_avg['Measure_pretty'].isin(bottom5_measures)]

# Create horizontal bar chart with hospital colors
fig3 = px.bar(bottom5_data,
              y='Measure_pretty',
              x='Score',
              color='Hospital_Short',
              orientation='h',
              title='📉 Bottom 5 Patient Satisfaction Measures (Need Improvement)',
              labels={'Score': 'Average Score', 'Measure_pretty': ''},
              color_discrete_map={v: HOSPITAL_COLORS[k] for k, v in HOSPITAL_SHORT_NAMES.items()},
              barmode='group',
              height=500)

fig3.update_layout(
    title_font_size=20,
    title_x=0.5,
    font=dict(size=11),
    legend=dict(title="Hospital"),
    xaxis_title='Average Score'
)

fig3.update_traces(texttemplate='%{x:.1f}', textposition='outside')

fig3.show()
save_chart(fig3, "03_bottom5_hcahps")
print("✅ Chart 3 complete!\n")


# ============================================================
# 4️⃣ HCAHPS TREEMAP (Top 5 by Hospital)
# Interactive visualization of each hospital's top measures
# ============================================================

print("=" * 70)
print("🟩 Creating Chart 4: HCAHPS Treemap (Top 5 per Hospital)")
print("=" * 70)

# Get top 5 measures per hospital
top5_per_hospital = (hcahps_data.groupby(['Hospital_Short', 'Measure_pretty'])['Score']
                     .mean()
                     .reset_index()
                     .sort_values(['Hospital_Short', 'Score'], ascending=[True, False])
                     .groupby('Hospital_Short')
                     .head(5))

# Create treemap
fig4 = px.treemap(top5_per_hospital,
                  path=['Hospital_Short', 'Measure_pretty'],
                  values='Score',
                  color='Score',
                  color_continuous_scale='RdYlGn',
                  title='🟩 Top 5 Patient Satisfaction Measures by Hospital',
                  height=600)

fig4.update_layout(
    title_font_size=20,
    title_x=0.5,
    font=dict(size=11)
)

fig4.update_traces(textposition='middle center',
                   textfont_size=10)

fig4.show()
save_chart(fig4, "04_treemap_top5")
print("✅ Chart 4 complete!\n")


# ============================================================
# 5️⃣ CORRELATION HEATMAP
# Shows relationships between Readmission, Safety, and HCAHPS
# ============================================================

print("=" * 70)
print("🔥 Creating Chart 5: Performance Correlation Heatmap")
print("=" * 70)

# Prepare correlation data from clean dataframe
corr_data = clean[['Readmission_Score_mean', 'HCAHPS_LinearMean_mean', 'Safety_Score_mean']].copy()
corr_data.columns = ['Readmissions', 'Patient Experience', 'Safety']

# Calculate correlation matrix
correlation_matrix = corr_data.corr()

# Create heatmap
fig5 = px.imshow(correlation_matrix,
                 labels=dict(color="Correlation"),
                 x=['Readmissions', 'Patient Experience', 'Safety'],
                 y=['Readmissions', 'Patient Experience', 'Safety'],
                 color_continuous_scale='RdBu_r',
                 aspect="auto",
                 title='🔥 Performance Metrics Correlation Heatmap',
                 text_auto='.2f',
                 height=500)

fig5.update_layout(
    title_font_size=20,
    title_x=0.5,
    font=dict(size=12),
    xaxis_title='',
    yaxis_title=''
)

fig5.update_xaxes(side="bottom")
fig5.show()
save_chart(fig5, "05_correlation_heatmap")
print("✅ Chart 5 complete!\n")


# ============================================================
# SUMMARY STATISTICS
# ============================================================

print("=" * 70)
print("📊 VISUALIZATION SUMMARY")
print("=" * 70)
print(f"✅ All 5 charts created successfully!")
print(f"\n📋 Data Overview:")
print(f"   • Total hospitals analyzed: {len(clean)}")
print(f"   • Total measures: {long['Measure'].nunique()}")
print(f"   • HCAHPS measures: {len(hcahps_data['Measure'].unique())}")
print(f"   • Total data points: {len(long)}")
print("=" * 70)

✅ Plotly configured!

🎨 Hospital Color Coding:
   #FF6B6B ● Manatee Memorial
   #4ECDC4 ● Sarasota Memorial
   #45B7D1 ● Blake Hospital
   #FFA07A ● Lakewood Ranch

📊 Creating Chart 1: Overall Hospital Comparison


💾 Saved: 01_overall_comparison.html, 01_overall_comparison.png
✅ Chart 1 complete!

📈 Creating Chart 2: Top 5 HCAHPS Questions


💾 Saved: 02_top5_hcahps.html, 02_top5_hcahps.png
✅ Chart 2 complete!

📉 Creating Chart 3: Bottom 5 HCAHPS Questions


💾 Saved: 03_bottom5_hcahps.html, 03_bottom5_hcahps.png
✅ Chart 3 complete!

🟩 Creating Chart 4: HCAHPS Treemap (Top 5 per Hospital)


💾 Saved: 04_treemap_top5.html, 04_treemap_top5.png
✅ Chart 4 complete!

🔥 Creating Chart 5: Performance Correlation Heatmap


💾 Saved: 05_correlation_heatmap.html, 05_correlation_heatmap.png
✅ Chart 5 complete!

📊 VISUALIZATION SUMMARY
✅ All 5 charts created successfully!

📋 Data Overview:
   • Total hospitals analyzed: 4
   • Total measures: 119
   • HCAHPS measures: 93
   • Total data points: 476


Hospitals strenghts & Opportunities

In [25]:
"""
Hospital Performance Summary: Strengths, Opportunities & Key Findings
Complete analysis with actionable insights 🏥
"""

import pandas as pd

print("=" * 80)
print("🏥 HOSPITAL PERFORMANCE ANALYSIS REPORT")
print("Bradenton-Sarasota Area Hospital Comparison")
print("=" * 80)
print()

# ============================================================
# PART 1: INDIVIDUAL HOSPITAL SUMMARIES
# ============================================================

# Filter HCAHPS measures only
hcahps_data = long[long['Measure'].str.startswith('H_', na=False)].copy()

# Group by hospital and measure to get average scores
hospital_measures = hcahps_data.groupby(['Hospital Name', 'Measure_pretty'])['Score'].mean().reset_index()

# Get list of hospitals (sorted alphabetically)
hospitals = sorted(hospital_measures['Hospital Name'].unique())

# Analyze each hospital
for hospital in hospitals:
    print(f"--- {hospital} ---")
    print()
    
    # Filter data for this hospital
    hospital_data = hospital_measures[hospital_measures['Hospital Name'] == hospital].copy()
    hospital_data = hospital_data.sort_values('Score', ascending=False)
    
    # Get top 5 and bottom 5
    top_5 = hospital_data.head(5)
    bottom_5 = hospital_data.tail(5)
    
    # STRENGTHS
    print("Strengths (Top 5 HCAHPS Measures):")
    for idx, row in top_5.iterrows():
        measure = row['Measure_pretty']
        score = row['Score']
        print(f"  - {measure}: {score:.2f}")
    
    print()
    
    # OPPORTUNITIES
    print("Opportunities (Bottom 5 HCAHPS Measures):")
    for idx, row in bottom_5.iterrows():
        measure = row['Measure_pretty']
        score = row['Score']
        print(f"  - {measure}: {score:.2f}")
    
    print("-" * 80)
    print()


# ============================================================
# PART 2: KEY FINDINGS & INSIGHTS
# ============================================================

print("\n" + "=" * 80)
print("📊 DATA ANALYSIS: KEY FINDINGS & INSIGHTS")
print("=" * 80)
print()

# Calculate overall statistics
overall_avg = hospital_measures.groupby('Hospital Name')['Score'].mean().sort_values(ascending=False)
overall_comparison = clean[['Hospital Name', 'Readmission_Score_mean', 
                           'HCAHPS_LinearMean_mean', 'Safety_Score_mean']].copy()

print("1️⃣ OVERALL HOSPITAL PERFORMANCE RANKINGS")
print("-" * 80)
for idx, (hospital, score) in enumerate(overall_avg.items(), 1):
    medal = "🥇" if idx == 1 else "🥈" if idx == 2 else "🥉" if idx == 3 else f"  {idx}."
    print(f"{medal} {hospital}")
    print(f"    Average HCAHPS Score: {score:.2f}")
print()

print("2️⃣ PERFORMANCE BY CATEGORY")
print("-" * 80)
for _, row in overall_comparison.iterrows():
    hospital = row['Hospital Name']
    print(f"\n{hospital}:")
    print(f"  • Readmissions: {row['Readmission_Score_mean']:.2f}")
    print(f"  • Patient Experience (HCAHPS): {row['HCAHPS_LinearMean_mean']:.2f}")
    print(f"  • Safety: {row['Safety_Score_mean']:.2f}")
print()

print("\n3️⃣ COMMON STRENGTHS ACROSS ALL HOSPITALS")
print("-" * 80)
# Find measures that score high across all hospitals
all_hospitals_top = hcahps_data.groupby('Measure_pretty')['Score'].mean().nlargest(5)
print("These patient satisfaction areas are performing well regionally:")
for measure, score in all_hospitals_top.items():
    print(f"  ✅ {measure} (Average: {score:.2f})")
print()

print("4️⃣ COMMON OPPORTUNITIES ACROSS ALL HOSPITALS")
print("-" * 80)
# Find measures that score low across all hospitals
all_hospitals_bottom = hcahps_data.groupby('Measure_pretty')['Score'].mean().nsmallest(5)
print("These areas need regional improvement focus:")
for measure, score in all_hospitals_bottom.items():
    print(f"  ⚠️  {measure} (Average: {score:.2f})")
print()

print("5️⃣ PERFORMANCE GAPS & DISPARITIES")
print("-" * 80)
# Calculate variation in scores for each measure
measure_variation = hcahps_data.groupby('Measure_pretty')['Score'].agg(['min', 'max', 'std']).reset_index()
measure_variation['gap'] = measure_variation['max'] - measure_variation['min']
largest_gaps = measure_variation.nlargest(5, 'gap')

print("Measures with largest performance gaps between hospitals:")
for _, row in largest_gaps.iterrows():
    print(f"  • {row['Measure_pretty']}")
    print(f"    Range: {row['min']:.2f} to {row['max']:.2f} (Gap: {row['gap']:.2f} points)")
print()

print("6️⃣ CORRELATION INSIGHTS")
print("-" * 80)
# Correlation analysis
corr_data = clean[['Readmission_Score_mean', 'HCAHPS_LinearMean_mean', 'Safety_Score_mean']].copy()
corr_data.columns = ['Readmissions', 'Patient_Experience', 'Safety']
correlation_matrix = corr_data.corr()

print("Relationship between performance metrics:")
print(f"  • Readmissions ↔ Patient Experience: {correlation_matrix.loc['Readmissions', 'Patient_Experience']:.2f}")
print(f"  • Readmissions ↔ Safety: {correlation_matrix.loc['Readmissions', 'Safety']:.2f}")
print(f"  • Patient Experience ↔ Safety: {correlation_matrix.loc['Patient_Experience', 'Safety']:.2f}")
print()

# Interpret strongest correlation
correlations = [
    ('Readmissions', 'Patient_Experience', correlation_matrix.loc['Readmissions', 'Patient_Experience']),
    ('Readmissions', 'Safety', correlation_matrix.loc['Readmissions', 'Safety']),
    ('Patient_Experience', 'Safety', correlation_matrix.loc['Patient_Experience', 'Safety'])
]
strongest = max(correlations, key=lambda x: abs(x[2]))

if abs(strongest[2]) > 0.7:
    strength = "STRONG"
elif abs(strongest[2]) > 0.4:
    strength = "MODERATE"
else:
    strength = "WEAK"

direction = "positive" if strongest[2] > 0 else "negative"
print(f"💡 Key Insight: {strength} {direction} correlation between {strongest[0]} and {strongest[1]}")
if strongest[2] > 0:
    print(f"   → Hospitals with better {strongest[0]} tend to have better {strongest[1]}")
else:
    print(f"   → As {strongest[0]} improves, {strongest[1]} tends to decline")
print()


# ============================================================
# PART 3: NEXT STEPS & RECOMMENDATIONS
# ============================================================

print("\n" + "=" * 80)
print("🎯 NEXT STEPS & RECOMMENDATIONS")
print("=" * 80)
print()

print("FOR HOSPITAL ADMINISTRATORS:")
print("-" * 80)

# Generate hospital-specific recommendations
for hospital in hospitals:
    hospital_data = hospital_measures[hospital_measures['Hospital Name'] == hospital]
    bottom_3 = hospital_data.nsmallest(3, 'Score')
    
    print(f"\n📋 {hospital}:")
    
    recommendations = []
    for _, row in bottom_3.iterrows():
        measure = row['Measure_pretty'].lower()
        
        if 'communication' in measure and 'medicine' in measure:
            recommendations.append("Enhance medication counseling protocols and pharmacist involvement")
        elif 'communication' in measure and 'nurse' in measure:
            recommendations.append("Implement nurse communication training and bedside rounds")
        elif 'communication' in measure and 'doctor' in measure:
            recommendations.append("Improve physician-patient communication through training")
        elif 'quiet' in measure:
            recommendations.append("Implement quiet hours and noise reduction initiatives")
        elif 'response' in measure or 'call' in measure:
            recommendations.append("Review staffing ratios and response time protocols")
        elif 'transition' in measure or 'discharge' in measure:
            recommendations.append("Strengthen care transition and discharge planning processes")
        elif 'clean' in measure:
            recommendations.append("Enhance environmental services and cleaning schedules")
    
    # Remove duplicates and show top 3
    recommendations = list(dict.fromkeys(recommendations))[:3]
    for i, rec in enumerate(recommendations, 1):
        print(f"  {i}. {rec}")

print("\n\nFOR REGIONAL HEALTHCARE PLANNING:")
print("-" * 80)
print("  1. Collaborate on regional best practices for communication about medications")
print("  2. Share successful strategies for maintaining quiet hospital environments")
print("  3. Develop standardized care transition protocols across all facilities")
print("  4. Create peer learning networks among hospitals for quality improvement")
print("  5. Invest in patient experience training programs regionally")

print("\n\nFOR FUTURE RESEARCH:")
print("-" * 80)
print("  1. Investigate root causes of low medication communication scores")
print("  2. Analyze relationship between staffing levels and responsiveness scores")
print("  3. Study impact of noise reduction interventions on patient satisfaction")
print("  4. Examine care transition programs at top-performing hospitals")
print("  5. Conduct patient interviews to understand improvement priorities")

print("\n\nFOR DATA ANALYSIS CONTINUATION:")
print("-" * 80)
print("  1. Track trends over time (quarterly/annual comparisons)")
print("  2. Segment analysis by patient demographics or medical conditions")
print("  3. Compare performance against state and national benchmarks")
print("  4. Analyze readmission rates by specific diagnosis categories")
print("  5. Correlate patient volume with quality metrics")

print("\n\n" + "=" * 80)
print("✅ ANALYSIS COMPLETE")
print("=" * 80)
print(f"\n📊 Summary Statistics:")
print(f"   • Hospitals Analyzed: {len(hospitals)}")
print(f"   • Total HCAHPS Measures: {hcahps_data['Measure'].nunique()}")
print(f"   • Total Data Points: {len(hcahps_data)}")
print(f"   • Average Regional HCAHPS Score: {hcahps_data['Score'].mean():.2f}")
print(f"   • Regional Score Range: {hcahps_data['Score'].min():.2f} - {hcahps_data['Score'].max():.2f}")
print("\n" + "=" * 80)

🏥 HOSPITAL PERFORMANCE ANALYSIS REPORT
Bradenton-Sarasota Area Hospital Comparison

--- HCA FLORIDA BLAKE HOSPITAL ---

Strengths (Top 5 HCAHPS Measures):
  - Nurse Communication (Score): 85.00
  - Doctor Communication (Score): 84.00
  - Hospital Cleanliness (Score): 82.00
  - Discharge Information (Score): 80.00
  - Overall Hospital Rating (Score): 79.00

Opportunities (Bottom 5 HCAHPS Measures):
  - Understood Symptoms to Watch - Yes: nan
  - Would Recommend Hospital (Stars): nan
  - Would Recommend Hospital - Definitely Yes: nan
  - Would Recommend Hospital - Definitely/Probably No: nan
  - Would Recommend Hospital - Probably Yes: nan
--------------------------------------------------------------------------------

--- LAKEWOOD RANCH MEDICAL CENTER ---

Strengths (Top 5 HCAHPS Measures):
  - Nurse Communication (Score): 90.00
  - Doctor Communication (Score): 87.00
  - Hospital Cleanliness (Score): 87.00
  - Discharge Information (Score): 86.00
  - Would Recommend Hospital (Score): 

In [20]:
pip install kaleido


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: C:\Users\dorvi\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip
